In [1]:
import os
import sys
import time
import pandas as pd
from dotenv import load_dotenv
import json
from IPython.display import display

# --- 1. PATH RESOLUTION & IMPORTS ---
current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)

load_dotenv()

from workflow_engine.orchestrators.dynamic_graph import build_dynamic_pipeline
from workflow_engine.utils.data_profiler import extract_dataset_metadata
from workflow_engine.utils.llm_judge import grade_report_with_llm

# --- 2. EXPERIMENTAL MATRIX CONFIGURATION ---
datasets_config = {
    "Heart_Disease_1K": {
        "path": os.path.join(project_root, "data", "raw", "heart_disease_statlog.csv").replace('\\', '/'),
        "target": "num"
    },
    "Telco_Churn_7K": {
        "path": os.path.join(project_root, "data", "raw", "telco_customer_churn.csv").replace('\\', '/'),
        "target": "Churn"
    },
    "PaySim_Fraud_100K": {
        "path": os.path.join(project_root, "data", "raw", "paysim_100k_sample.csv").replace('\\', '/'),
        "target": "isFraud"
    }
}

trigger_prompts = {
    "RAPID_BASELINE": "I have a tabular dataset and I need a baseline classification model as quickly as possible. I only care about getting a reasonable benchmark with minimal computation. Please avoid unnecessary processing and give me a quick performance summary.",
    "QUICK_EXPLAINABLE": "I need a predictive model that I can easily explain to non-technical stakeholders. The model should prioritise interpretability over predictive performance, and I don't want complex ensembles or neural networks.",
    "ENTERPRISE_STANDARD": "Build a production-ready machine learning pipeline for this dataset. I want a reliable end-to-end workflow including data cleaning, exploratory analysis, feature engineering, model training, evaluation and a comprehensive report suitable for deployment discussions.",
    "KAGGLE_COMPETITOR": "Maximise predictive performance on this dataset. Computational cost is not important. Train strong machine learning models, compare them thoroughly, and select whichever achieves the highest predictive performance regardless of interpretability.",
    "REGULATORY_COMPLIANCE": "I need a machine learning solution for a highly regulated environment. Every prediction should be easy to justify and audit, so transparency and explainability are much more important than maximising accuracy.",
    "C_SUITE_PITCH": "I don't need a predictive model. I just want an executive-level understanding of this dataset. Produce exploratory analysis, key insights and visualisations that I can present to senior management without discussing machine learning algorithms."
}

# --- 3. EXECUTION ENGINE ---
evaluation_results = []
app = build_dynamic_pipeline()

print("🚀 Starting the Automated 18-Run Matrix...\n" + "="*60)

for dataset_name, d_config in datasets_config.items():
    raw_path = d_config["path"]
    target_col = d_config["target"]
    
    # Pre-Flight Profiling
    dataset_metadata = extract_dataset_metadata(raw_path, target_col)
    
    for expected_preset, user_request in trigger_prompts.items():
        print(f"\n🧪 RUNNING: [{dataset_name}] | [{expected_preset}]")
        
        initial_state = {
            "messages": [f"Automated Eval: {dataset_name} - {expected_preset}"],
            "user_request": user_request,
            "target_variable": target_col,
            "raw_dataset_path": raw_path,
            "current_dataset_path": raw_path,
            "artifacts": {},
            "current_step": "start",
            "error_flag": False,
            "error_message": "",
            "dataset_metadata": dataset_metadata,
            "active_preset": "",
            "proposed_route": [],
            "z3_verification_status": False,
            "total_sleep_time": 0.0,
            "supervisor_latency": 0.0,
            "supervisor_tokens": 0,
            "supervisor_calls": 0,
            "total_input_tokens": 0,
            "total_output_tokens": 0,
            "api_call_timestamps": []
        }

        start_time = time.time()
        final_state = initial_state
        executed_route = []
        
        try:
            # Use invoke to keep your metric tracking logic consistent
            final_state = app.invoke(initial_state, {"recursion_limit": 25})
            # To get executed route from final state if nodes were tracked
            executed_route = final_state.get("proposed_route", []) 
        except Exception as e:
            final_state["error_flag"] = True
            final_state["error_message"] = str(e)

        end_time = time.time()
        gross_runtime = end_time - start_time
        pure_compute_time = gross_runtime - final_state.get("total_sleep_time", 0.0)
        
        # --- Run OpenAI Judge ---
        judge_scores = {"Technical Score": 0, "Actionability Score": 0, "Alignment Score": 0}
        if not final_state.get("error_flag") and "final_report" in final_state.get("artifacts", {}):
            report_path = final_state["artifacts"]["final_report"]
            if os.path.exists(report_path):
                with open(report_path, "r", encoding="utf-8") as f:
                    judge_scores = grade_report_with_llm(f.read(), expected_preset, user_request)

        # --- Extract Model Metrics (If Applicable) ---
        model_metrics = {
            "Model Name": "N/A",
            "Accuracy": "N/A",
            "Precision": "N/A",
            "Recall": "N/A",
            "F1 Score": "N/A",
            "Training Time (s)": "N/A"
        }
        
        # We look for the model metrics JSON in the artifacts dictionary
        if not final_state.get("error_flag") and "model_metrics" in final_state.get("artifacts", {}):
            metrics_path = final_state["artifacts"]["model_metrics"]
            if os.path.exists(metrics_path):
                with open(metrics_path, "r", encoding="utf-8") as f:
                    try:
                        metrics_data = json.load(f)
                        model_metrics["Model Name"] = metrics_data.get("model_name", "N/A")
                        # Round floats to 4 decimal places for clean tables
                        model_metrics["Accuracy"] = round(metrics_data.get("accuracy", 0.0), 4)
                        model_metrics["Precision"] = round(metrics_data.get("precision", 0.0), 4)
                        model_metrics["Recall"] = round(metrics_data.get("recall", 0.0), 4)
                        model_metrics["F1 Score"] = round(metrics_data.get("f1_score", 0.0), 4)
                        model_metrics["Training Time (s)"] = round(metrics_data.get("model_training_time", 0.0), 4)
                    except json.JSONDecodeError:
                        print(f"   ⚠️ Could not parse model_metrics.json at {metrics_path}")
        
        # --- Calculate Worker vs. Global Metrics (NEW) ---
        # Get raw totals
        total_in_tokens = final_state.get("total_input_tokens", 0)
        total_out_tokens = final_state.get("total_output_tokens", 0)
        total_tokens = total_in_tokens + total_out_tokens
        
        timestamps = final_state.get("api_call_timestamps", [])
        total_api_calls = len(timestamps)
        
        supervisor_tokens = final_state.get("supervisor_tokens", 0)
        supervisor_calls = final_state.get("supervisor_calls", 0)
        supervisor_latency = final_state.get("supervisor_latency", 0.0)
        
        # Calculate Worker specific costs
        worker_tokens = total_tokens - supervisor_tokens
        worker_calls = total_api_calls - supervisor_calls
        worker_compute_time = pure_compute_time - supervisor_latency

        # --- Append to Master List (UPDATED) ---
        evaluation_results.append({
            "Dataset": dataset_name,
            "Preset": expected_preset,
            "Executed Route": " -> ".join(executed_route),
            
            # Global Time Metrics
            "Runtime (s)": round(pure_compute_time, 2),
            
            # Orchestration Tax vs Worker Compute
            "Supervisor Latency (s)": round(supervisor_latency, 2),
            "Worker Compute (s)": round(worker_compute_time, 2),
            "Supervisor Tokens": supervisor_tokens,
            "Worker Tokens": worker_tokens,
            "Total Tokens": total_tokens,
            "Supervisor API Calls": supervisor_calls,
            "Worker API Calls": worker_calls,
            "Total API Calls": total_api_calls,
            
            # Predictive Metrics
            "Model Name": model_metrics["Model Name"],
            "Accuracy": model_metrics["Accuracy"],
            "Precision": model_metrics["Precision"],
            "Recall": model_metrics["Recall"],
            "F1 Score": model_metrics["F1 Score"],
            "Training Time (s)": model_metrics["Training Time (s)"],
            
            # Interpretability Metrics
            "Judge: Tech": judge_scores.get("Technical Score", 0),
            "Judge: Action": judge_scores.get("Actionability Score", 0),
            "Judge: Align": judge_scores.get("Alignment Score", 0)
        })
        
        print(f"   [DONE] Route: {' -> '.join(executed_route)} | F1: {model_metrics['F1 Score']} | Train Time: {model_metrics['Training Time (s)']}s | Align Score: {judge_scores.get('Alignment Score', 0)}/10")

# --- 4. EXPORT TO reports/results ---
df_results = pd.DataFrame(evaluation_results)
results_dir = os.path.join(project_root, "reports", "results")
os.makedirs(results_dir, exist_ok=True)
df_results.to_csv(os.path.join(results_dir, "eval_results.csv"), index=False)
print(f"\n💾 Results exported to {results_dir}")
display(df_results)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


🚀 Starting the Automated 18-Run Matrix...

🧪 RUNNING: [Heart_Disease_1K] | [RAPID_BASELINE]

🤖 [Supervisor] Analyzing state to determine next step...
🧠 [Planner] No verified route found. Invoking LLM and Z3 Solver...
⏳ [Rate Limit Protocol] Pausing for 5.0s...
🎯 [Planner] Attempt 1: Classified as RAPID_BASELINE
🛤️ [Planner] Proposed Path: ['data_cleaning', 'feature_engineering', 'modelling', 'reporting']
✅ [Z3] Route formally verified as SAFE.
--- AGENT: DATA CLEANING ---

--- ATTEMPT 1/3 ---


Python REPL can execute arbitrary code. Use with caution.


Generated Code:
 import pandas as pd
import os

output_dir = r'd:/ArtificialIntelligence/data-science-workflow/data/processed'
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(r'd:/ArtificialIntelligence/data-science-workflow/data/raw/heart_disease_statlog.csv')

target_keywords = ['charge', 'price', 'balance', 'amount', 'fee']
for col in df.columns:
    if df[col].dtype == 'object':
        for keyword in target_keywords:
            if keyword in col.lower():
                df[col] = pd.to_numeric(df[col], errors='coerce')
                break

# Drop unstructured free-text columns (assuming high cardinality object columns are text)
for col in df.columns:
    if df[col].dtype == 'object':
        if df[col].nunique() > 50:
            df = df.drop(columns=[col])

# Drop columns with > 50% missing values
threshold = len(df) * 0.5
for col in df.columns:
    if df[col].isnull().sum() > threshold:
        df = df.drop(columns=[col])

# Impute missing values
for col in df.column

<string>:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


...

d:\venvs\data-science-workflow\lib\site-packages\xgboost\training.py:200: UserWarning: [05:10:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Status: Success

🤖 [Supervisor] Analyzing state to determine next step...
⚡ [Dispatcher] Using pre-verified route: ['data_cleaning', 'eda', 'feature_engineering', 'modelling', 'reporting']
--- AGENT: REPORTING ---
--- PRESET: ENTERPRISE_STANDARD ---
Status: Success - Report saved to d:/ArtificialIntelligence/data-science-workflow/reports/final_reports/final_report.md

🤖 [Supervisor] Analyzing state to determine next step...
⚡ [Dispatcher] Using pre-verified route: ['data_cleaning', 'eda', 'feature_engineering', 'modelling', 'reporting']
⚖️  [OpenAI Judge] Grading report against the strict ENTERPRISE_STANDARD rubric...
   [DONE] Route: data_cleaning -> eda -> feature_engineering -> modelling -> reporting | F1: 0.7778 | Train Time: 2.9776s | Align Score: 9/10

🧪 RUNNING: [PaySim_Fraud_100K] | [KAGGLE_COMPETITOR]

🤖 [Supervisor] Analyzing state to determine next step...
🧠 [Planner] No verified route found. Invoking LLM and Z3 Solver...
⏳ [Rate Limit Protocol] Pausing for 5.0s...
🎯 [Planne

d:\venvs\data-science-workflow\lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=15. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
d:\venvs\data-science-workflow\lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=15. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
d:\venvs\data-science-workflow\lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=15. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
d:\venvs\data-science-workflow\lib\site-packages\xgboost\training.py:200: UserWarning: [05:12:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Status: Success

🤖 [Supervisor] Analyzing state to determine next step...
⚡ [Dispatcher] Using pre-verified route: ['data_cleaning', 'eda', 'feature_engineering', 'modelling', 'reporting']
--- AGENT: REPORTING ---
--- PRESET: KAGGLE_COMPETITOR ---
Status: Success - Report saved to d:/ArtificialIntelligence/data-science-workflow/reports/final_reports/final_report.md

🤖 [Supervisor] Analyzing state to determine next step...
⚡ [Dispatcher] Using pre-verified route: ['data_cleaning', 'eda', 'feature_engineering', 'modelling', 'reporting']
⚖️  [OpenAI Judge] Grading report against the strict KAGGLE_COMPETITOR rubric...
   [DONE] Route: data_cleaning -> eda -> feature_engineering -> modelling -> reporting | F1: 0.8936 | Train Time: 110.3615s | Align Score: 9/10

🧪 RUNNING: [PaySim_Fraud_100K] | [REGULATORY_COMPLIANCE]

🤖 [Supervisor] Analyzing state to determine next step...
🧠 [Planner] No verified route found. Invoking LLM and Z3 Solver...
⏳ [Rate Limit Protocol] Pausing for 5.0s...
🎯 [Plan

,Dataset,Preset,Executed Route,Runtime (s),Supervisor Latency (s),Worker Compute (s),Supervisor Tokens,Worker Tokens,Total Tokens,Supervisor API Calls,...,Total API Calls,Model Name,Accuracy,Precision,Recall,F1 Score,Training Time (s),Judge: Tech,Judge: Action,Judge: Align
0,Heart_Disease_1K,RAPID_BASELINE,data_cleaning -> feature_engineering -> modell...,7.92,1.12,6.80,578,4411,4989,1,...,5,LogisticRegression,0.8152,0.8152,0.8152,0.8152,0.0,9,8,10
1,Heart_Disease_1K,QUICK_EXPLAINABLE,data_cleaning -> feature_engineering -> modell...,9.62,0.93,8.69,592,4868,5460,1,...,5,DecisionTreeClassifier,0.8152,0.8188,0.8152,0.8162,0.0435,9,8,9
2,Heart_Disease_1K,ENTERPRISE_STANDARD,data_cleaning -> eda -> feature_engineering ->...,24.39,0.96,23.44,603,9089,9692,1,...,6,RandomForestClassifier,0.875,0.8774,0.875,0.8755,2.0186,9,8,9
3,Heart_Disease_1K,KAGGLE_COMPETITOR,data_cleaning -> eda -> feature_engineering ->...,40.41,0.86,39.55,599,9929,10528,1,...,6,XGBClassifier,0.875,0.8749,0.875,0.8749,23.8085,9,8,9
4,Heart_Disease_1K,REGULATORY_COMPLIANCE,data_cleaning -> eda -> feature_engineering ->...,11.64,0.91,10.73,593,7197,7790,1,...,6,DecisionTreeClassifier,0.8539,0.8534,0.8539,0.8532,0.0187,9,9,10
5,Heart_Disease_1K,C_SUITE_PITCH,data_cleaning -> eda -> reporting,8.06,0.83,7.23,592,4565,5157,1,...,4,N/A,N/A,N/A,N/A,N/A,N/A,9,8,9
6,Telco_Churn_7K,RAPID_BASELINE,data_cleaning -> feature_engineering -> modell...,8.14,1.69,6.45,694,4398,5092,1,...,5,LogisticRegression,0.8,0.6134,0.5455,0.5774,0.0086,9,8,10
7,Telco_Churn_7K,QUICK_EXPLAINABLE,data_cleaning -> feature_engineering -> modell...,8.70,0.85,7.85,595,4685,5280,1,...,5,DecisionTreeClassifier,0.7851,0.772,0.7851,0.7758,0.0135,9,10,10
8,Telco_Churn_7K,ENTERPRISE_STANDARD,data_cleaning -> eda -> feature_engineering ->...,21.14,9.77,11.37,606,8085,8691,1,...,6,LogisticRegression,0.8014,0.6144,0.5568,0.5842,0.4573,9,8,9
9,Telco_Churn_7K,KAGGLE_COMPETITOR,data_cleaning -> eda -> feature_engineering ->...,59.92,0.94,58.98,600,9403,10003,1,...,6,LogisticRegression,0.7573,0.529,0.7608,0.624,46.5558,9,8,9
